In [0]:
from pyspark.sql import functions as F

In [0]:
SOURCE_CATALOG_NAME = 'beverage_sales'
SOURCE_SCHEMA_NAME = 'silver'
SOURCE_TABLE_NAME = 'sales'

TARGET_CATALOG_NAME = 'beverage_sales'
TARGET_SCHEMA_NAME = 'gold'
TARGET_TABLE_NAME = 'dim_region'

UNKNOWN_KEY = -1

In [0]:
df_unknown_member = spark.createDataFrame(
    [(UNKNOWN_KEY, 'UNKNOWN')],
    'region_key bigint, region string'
)

In [0]:
df_dim_region = (
    spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.{SOURCE_TABLE_NAME}')
    .select('region')
    .dropDuplicates(['region'])
    .withColumn('region_key', F.abs(F.xxhash64(F.col('region'))))
    .select(
        'region_key',
        'region'
    )
    .unionByName(df_unknown_member)
)

In [0]:
df_dim_region\
    .write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')